# Load Config and Raw Annotation File

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
from pathlib import Path

import pandas as pd

# Define the path to the main SurgSigma annotation file
ANNOTATIONS_PATH = Path("/content/drive/MyDrive/Surgical-VLM/data/SurgSigma/SurgSigma_v0.1_T50subset.json")

# The SurgSigma directory is the parent directory of the annotation file
SURGSIGMA_DIR = ANNOTATIONS_PATH.parent

# Load the SurgSigma annotations directly into the 'surgsigma' variable
with open(ANNOTATIONS_PATH) as f:
    surgsigma = json.load(f)

print("Top-level keys:", list(surgsigma.keys()))
for key in ["meta data", "tasks", "images", "videos", "annos"]:
    val = surgsigma.get(key)
    n = len(val) if isinstance(val, list) else "n/a (not a list)"
    print(f"  {key:12s} -> {n}")

Top-level keys: ['meta data', 'tasks', 'images', 'videos', 'annos']
  meta data    -> n/a (not a list)
  tasks        -> 18
  images       -> 85676
  videos       -> 50
  annos        -> 1345331


# Build Lookups

In [ ]:
tasks_df = pd.DataFrame(surgsigma["tasks"])
images_df = pd.DataFrame(surgsigma["images"])
videos_df = pd.DataFrame(surgsigma["videos"])
annos_df = pd.DataFrame(surgsigma["annos"])

print("tasks :", tasks_df.shape)
print("images:", images_df.shape)
print("videos:", videos_df.shape)
print("annos :", annos_df.shape)

display(tasks_df)

task_lookup = dict(zip(tasks_df["id"], tasks_df["name"])) if "id" in tasks_df.columns else {}
task_group_lookup = dict(zip(tasks_df["id"], tasks_df["group"])) if "group" in tasks_df.columns else {}

image_id_to_path = dict(zip(images_df["id"], images_df["image path"])) if "image path" in images_df.columns else {}
video_id_to_path = dict(zip(videos_df["id"], videos_df["video path"])) if len(videos_df) and "video path" in videos_df.columns else {}


tasks : (18, 3)
images: (85676, 6)
videos: (50, 6)
annos : (1345331, 11)


,id,name,group
0,0,Instrument Recognition,Understanding and Reasoning
1,1,Instrument Localization,Understanding and Reasoning
2,2,Instrument Segmentation,Understanding and Reasoning
3,3,Tissue and Organ Recognition,Understanding and Reasoning
4,4,Tissue and Organ Localization,Understanding and Reasoning
5,5,Phase Recognition,Understanding and Reasoning
6,6,Step Recognition,Understanding and Reasoning
7,7,Action Recognition,Understanding and Reasoning
8,8,Triplet Recognition,Understanding and Reasoning
9,9,Depth Estimation,Understanding and Reasoning


# Confirm CholecT50-Subset Task Coverage


In [ ]:
OFFICIAL_18_TASKS = {
    "Understanding and Reasoning": [
        "Instrument Recognition", "Instrument Localization", "Instrument Segmentation",
        "Tissue and Organ Recognition", "Tissue and Organ Localization",
        "Phase Recognition", "Step Recognition", "Action Recognition",
        "Triplet Recognition", "Depth Estimation", "Safety Assessment",
        "Surgical Image Captioning", "Surgical Video Captioning",
    ],
    "Planning and Generation": [
        "Action Remaining Prediction", "Next Action Planning", "Desmoking",
        "Next Frame Prediction", "Conditional Surgical Video Generation",
    ],
}

present_tasks = set(tasks_df["name"]) if "name" in tasks_df.columns else set()
all_official = {t for group in OFFICIAL_18_TASKS.values() for t in group}

print("Present in this download:", len(present_tasks))
print("Missing from official 18-task list:", sorted(all_official - present_tasks))
print("Present but NOT in official list (unexpected — investigate if non-empty):",
      sorted(present_tasks - all_official))


Present in this download: 18
Missing from official 18-task list: []
Present but NOT in official list (unexpected — investigate if non-empty): []


# Classify Every Annotation Using the OFFICIAL Routing Logic


In [ ]:
def extract_dense_prediction_paths(value):
    paths = []
    if isinstance(value, str):
        if value.strip():
            paths.append(value)
        return paths
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str) and item.strip():
                paths.append(item)
            elif isinstance(item, dict):
                for key in ("image path", "path", "mask path", "pred path", "prediction path"):
                    v = item.get(key)
                    if isinstance(v, str) and v.strip():
                        paths.append(v)
                        break
    return paths


def classify_route(anno):
    dense_paths = extract_dense_prediction_paths(anno.get("dense prediction"))
    if len(dense_paths) > 0:
        return "dense"

    video_ids = anno.get("videos") if isinstance(anno.get("videos"), list) else (
        [anno["video"]] if isinstance(anno.get("video"), int) else []
    )
    if video_ids:
        return "video"

    image_ids = anno.get("images") if isinstance(anno.get("images"), list) else (
        [anno["image"]] if isinstance(anno.get("image"), int) else []
    )
    if image_ids:
        return "image"

    return "skipped"


annos_records = surgsigma["annos"]
routes = [classify_route(a) for a in annos_records]
annos_df["route"] = routes

print("Route counts (matches build_instruction_tuning_data.py's own reported stats):")
print(annos_df["route"].value_counts())


Route counts (matches build_instruction_tuning_data.py's own reported stats):
route
image    747052
dense    464201
video    134078
Name: count, dtype: int64


# Per-Task Classification Table


In [ ]:
def get_task_name(anno):
    tt = anno.get("task type")
    if isinstance(tt, list) and len(tt) > 0:
        tt = tt[0]
    if isinstance(tt, int):
        return task_lookup.get(tt, f"unknown_id_{tt}")
    if isinstance(tt, str):
        return tt
    return None

annos_df["task_name"] = [get_task_name(a) for a in annos_records]

route_table = (
    annos_df.groupby("task_name")["route"]
    .value_counts()
    .unstack(fill_value=0)
)
route_table["total"] = route_table.sum(axis=1)
route_table["text_answerable"] = route_table.get("image", 0) == route_table["total"]

display(route_table.sort_values("total", ascending=False))

print("\nTasks that are 100% image-routed (safe for CLIP/VLM text-based training):")
print(route_table[route_table["text_answerable"]].index.tolist())

print("\nTasks with ANY dense/video rows (need pixel or clip output — exclude or handle separately):")
print(route_table[~route_table["text_answerable"]].index.tolist())


route,dense,image,video,total,text_answerable
task_name,,,,,
Next Frame Prediction,297830,0,0,297830,False
Triplet Recognition,0,154214,79176,233390,False
Phase Recognition,0,91312,43695,135007,False
Safety Assessment,0,127305,0,127305,True
Instrument Recognition,0,124060,0,124060,True
Action Recognition,0,108525,2353,110878,False
Surgical Image Captioning,0,72942,0,72942,True
Desmoking,55983,0,0,55983,False
Depth Estimation,55194,0,0,55194,False



Tasks that are 100% image-routed (safe for CLIP/VLM text-based training):
['Instrument Localization', 'Instrument Recognition', 'Safety Assessment', 'Surgical Image Captioning', 'Tissue and Organ Recognition']

Tasks with ANY dense/video rows (need pixel or clip output — exclude or handle separately):
['Action Recognition', 'Action Remaining Prediction', 'Conditional Surgical Video Generation', 'Depth Estimation', 'Desmoking', 'Instrument Segmentation', 'Next Action Planning', 'Next Frame Prediction', 'Phase Recognition', 'Surgical Video Captioning', 'Triplet Recognition']


# Check for Multi-Image Annotations

In [ ]:
def n_images(anno):
    ids = anno.get("images") if isinstance(anno.get("images"), list) else (
        [anno["image"]] if isinstance(anno.get("image"), int) else []
    )
    return len(ids)

annos_df["n_images"] = [n_images(a) for a in annos_records]

image_routed = annos_df[annos_df["route"] == "image"]
multi_image = image_routed[image_routed["n_images"] > 1]

print(f"Image-routed annotations with >1 image: {len(multi_image)} / {len(image_routed)}")
if len(multi_image) > 0:
    print("\nBy task:")
    print(multi_image["task_name"].value_counts())
    print("\n  3_MasterDataset.ipynb's anno[\"images\"][0] is silently dropping images for these rows.")
else:
    print("\n Every image-routed annotation has exactly 1 image \u2014 anno['images'][0] is safe.")


Image-routed annotations with >1 image: 0 / 747052

✓ Every image-routed annotation has exactly 1 image — anno['images'][0] is safe.


# gt_label Structure Per Task

In [ ]:
pd.set_option("display.max_colwidth", 120)

target_tasks = route_table[route_table["text_answerable"]].index.tolist()

for task in target_tasks:
    sample = annos_df[annos_df["task_name"] == task].head(2)
    print(f"--- {task} ---")
    for _, row in sample.iterrows():
        orig = annos_records[row.name]
        print("question :", str(orig.get("question"))[:100])
        print("answer   :", str(orig.get("answer"))[:100])
        print("gt_label :", str(orig.get("gt label"))[:150])
        print()


--- Instrument Localization ---
question : Given the laparoscopic surgical image, locate all the tools in the format of bbox (x1,y1), (x2,y2).
answer   : hook (369,0),(877,567)
gt_label : ['hook (369,0),(877,567)']

question : Given the laparoscopic surgical image, locate all the tools in the format of bbox (x1,y1), (x2,y2).
answer   : hook (444,179),(931,675)
gt_label : ['hook (444,179),(931,675)']

--- Instrument Recognition ---
question : Given the laparoscopic cholecystectomy image, identify the surgical tool(s) present in the image.
answer   : hook
gt_label : ['hook']

question : Given the laparoscopic cholecystectomy image, identify the surgical tool(s) present in the image.
answer   : hook
gt_label : ['hook']

--- Safety Assessment ---
question : Given the laparoscopic cholecystectomy image, is the lower gallbladder detached from the liver bed?
answer   : no
gt_label : ['no']

question : Given the laparoscopic cholecystectomy image, is the lower gallbladder detached from the liv

In [ ]:
import time, shutil, subprocess
from pathlib import Path

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")

with open(CONFIG) as f:
    config = json.load(f)

DRIVE_CHOLECT50_ARCHIVE = Path("/content/drive/MyDrive/Surgical-VLM/data/CholecT50_raw.zip")  # adjust extension if .tar.gz
LOCAL_ARCHIVE_COPY = Path("/content/CholecT50_raw" + DRIVE_CHOLECT50_ARCHIVE.suffix)
LOCAL_CHOLECT50_DIR = Path("/content/CholecT50")

assert DRIVE_CHOLECT50_ARCHIVE.exists(), (
    f"Expected archive not found at {DRIVE_CHOLECT50_ARCHIVE}. "
    "Upload it to Drive first (see markdown above) before running this cell."
)

# FIX: extracting directly from the Drive-mounted archive means the zip/tar
# reader does many small random-access seeks against Drive's network
# filesystem -- very high latency per operation, which is what turned this
# into 29 minutes. Copy the single archive file to local disk first (one
# large sequential read, fast), then extract locally (local-to-local, fast
# regardless of file count).
print("Copying archive to local disk...")
t0 = time.time()
shutil.copy2(DRIVE_CHOLECT50_ARCHIVE, LOCAL_ARCHIVE_COPY)
print(f"Copy done in {time.time()-t0:.1f}s")

LOCAL_CHOLECT50_DIR.mkdir(parents=True, exist_ok=True)

# Native unzip/tar (compiled C, far faster than Python's zipfile/tarfile
# modules for extracting tens of thousands of small files) -q suppresses
# per-file printing, which also matters at this file count.
print("Extracting locally...")
t0 = time.time()
if LOCAL_ARCHIVE_COPY.suffix == ".zip":
    subprocess.run(["unzip", "-q", str(LOCAL_ARCHIVE_COPY), "-d", str(LOCAL_CHOLECT50_DIR)], check=True)
else:
    subprocess.run(["tar", "-xzf", str(LOCAL_ARCHIVE_COPY), "-C", str(LOCAL_CHOLECT50_DIR)], check=True)
print(f"Extract done in {time.time()-t0:.1f}s")

n_files = sum(1 for _ in LOCAL_CHOLECT50_DIR.rglob("*") if _.is_file())
print(f"\n{n_files:,} files extracted to {LOCAL_CHOLECT50_DIR}")

# Free the local zip copy now that it's extracted -- no need to keep both
LOCAL_ARCHIVE_COPY.unlink()

# Point config at the LOCAL extracted copy (not Drive) for actual training/reading
config["cholect50_dir"] = str(LOCAL_CHOLECT50_DIR)
with open(CONFIG, "w") as f:
    json.dump(config, f, indent=4)
print("config['cholect50_dir'] now points at local disk:", config["cholect50_dir"])
print()
print("This is LOCAL Colab disk -- it resets when the runtime disconnects.")
print("Re-run this cell at the start of every session before training.")


Copying archive to local disk...
Copy done in 2364.4s
Extracting locally...
Extract done in 1171.7s

100,918 files extracted to /content/CholecT50
config['cholect50_dir'] now points at local disk: /content/CholecT50

This is LOCAL Colab disk -- it resets when the runtime disconnects.
Re-run this cell at the start of every session before training.


In [ ]:
print(config["cholect50_dir"])

/content/CholecT50


# Image Existence Check

In [ ]:
from tqdm.auto import tqdm

CHOLECT50_DIR = Path(config["cholect50_dir"])

sample_check = images_df.sample(min(2000, len(images_df)), random_state=42)
exists_count = sum(
    (CHOLECT50_DIR / p).exists() if not Path(p).is_absolute() else Path(p).exists()
    for p in tqdm(sample_check["image path"])
)

print(f"{exists_count}/{len(sample_check)} sampled images resolve to real files "
      f"({100*exists_count/len(sample_check):.1f}%)")
print(f"CHOLECT50_DIR currently points to: {CHOLECT50_DIR}")
if exists_count == 0:
    print("\n 0% resolved — CHOLECT50_DIR is likely wrong, or you haven't placed the raw")
    print("   CholecT50 frames not there yet.")


  0%|          | 0/2000 [00:00<?, ?it/s]

1810/2000 sampled images resolve to real files (90.5%)
CHOLECT50_DIR currently points to: /content/CholecT50


# Save Reference Outputs

In [ ]:
PROCESSED_DIR = Path(config["processed_dir"])

route_table.to_csv(PROCESSED_DIR / "task_routing_reference.csv")
tasks_df.to_csv(PROCESSED_DIR / "tasks_lookup.csv", index=False)

print("Saved:")
print(PROCESSED_DIR / "task_routing_reference.csv")
print(PROCESSED_DIR / "tasks_lookup.csv")


Saved:
/content/drive/MyDrive/Surgical-VLM/processed/task_routing_reference.csv
/content/drive/MyDrive/Surgical-VLM/processed/tasks_lookup.csv
